<a href="https://colab.research.google.com/github/digitaldaimyo/AddressedStateAttention/blob/main/notebooks/asa_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ASA Training Notebook

Train **Addressed State Attention** models from scratch on FineWeb.

**Quick links:** [GitHub](https://github.com/digitaldaimyo/AddressedStateAttention) · [HF Models](https://huggingface.co/DigitalDaimyo/AddressedStateAttention) · [Paper](https://github.com/digitaldaimyo/AddressedStateAttention/tree/main/paper_drafts)

---

This notebook trains a small-to-medium ASA model (57M-187M params) on FineWeb in ~4-8 hours on a free Colab T4. For production runs, use A100.

In [ ]:
#@title Setup - Install & Imports
#@markdown Run this first. Installs ASA from GitHub and loads dependencies.

!pip install -q git+https://github.com/digitaldaimyo/AddressedStateAttention.git

import os
import math
import time
import pickle
from dataclasses import asdict
from typing import List, Tuple, Optional

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

# Check from asa package
try:
    from asa.training import ASMTrainConfig, build_model_from_cfg
    print("✓ ASA package loaded")
except ImportError:
    print("❌ Failed to import ASA. Check pip install succeeded.")
    raise

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Training will be very slow.")

torch.manual_seed(42)

In [ ]:

#@title Training Configuration
#@markdown Choose a preset or customize below.

# === Quick Presets ===
PRESET = "tiny"  #@param ["tiny", "small", "medium"] {type:"string"}

presets = {
    "tiny": {
        "embed_dim": 256, "num_layers": 12, "num_heads": 8,
        "num_slots": 16, "total_steps": 25_000, "tag": "asm_tiny"
    },
    "small": {
        "embed_dim": 384, "num_layers": 15, "num_heads": 8,
        "num_slots": 16, "total_steps": 50_000, "tag": "asm_small"
    },
    "medium": {
        "embed_dim": 768, "num_layers": 21, "num_heads": 12,
        "num_slots": 16, "total_steps": 75_000, "tag": "asm_medium"
    },
}

preset = presets[PRESET]

# === Model Config ===
cfg = ASMTrainConfig(
    # Data
    dataset_name="HuggingFaceFW/fineweb",
    dataset_config="sample-10BT",
    tokenizer_name="gpt2",
    max_seq_len=1024,

    # Model
    vocab_size=50257,
    embed_dim=preset["embed_dim"],
    num_layers=preset["num_layers"],
    num_heads=preset["num_heads"],
    num_slots=preset["num_slots"],
    slotspace_dim=8,
    mlp_ratio=4.0,
    dropout=0.1,

    # ASA settings
    use_content_read=True,
    use_slotspace_refine=True,
    use_rope_keys=True,
    use_alibi_write=True,

    # Training
    batch_size=32,
    learning_rate=3e-4,
    weight_decay=0.1,
    total_steps=preset["total_steps"],
    warmup_steps=1_000,
    grad_clip=1.0,
)

# === Training Workflow Params (not part of ASMTrainConfig) ===
MICRO_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
EVAL_INTERVAL = 1_000
LOG_INTERVAL = 100
SAVE_LAST_EVERY = 1_000
OUTPUT_DIR = "./asm_outputs"
TAG = preset["tag"]
CACHE_DIR = "./asm_cache"

# Optional: Google Drive paths (uncomment if mounted)
# OUTPUT_DIR = "./drive/MyDrive/asm_outputs"
# CACHE_DIR = "./drive/MyDrive/asm_cache"

# === Summary ===
model = build_model_from_cfg(cfg)
total_params = sum(p.numel() for p in model.parameters())

print(f"Config: {cfg.num_layers}L × {cfg.embed_dim}D, {cfg.num_slots} slots")
print(f"Params: {total_params/1e6:.2f}M")
print(f"Steps: {cfg.total_steps:,} (warmup: {cfg.warmup_steps:,})")
print(f"Batch: {cfg.batch_size} = {MICRO_BATCH_SIZE} × {GRAD_ACCUM_STEPS}")
print(f"Output: {OUTPUT_DIR}/{TAG}")

del model  # Free memory

In [ ]:

#@title Data Loading Utilities
#@markdown Tokenize and cache FineWeb dataset.

import struct
import numpy as np

# === Token stream builder ===
_MAGIC = b"TOKU32V1"
_HDR_STRUCT = struct.Struct("<8sIQ")

def _ensure_dir(path: str):
    d = os.path.dirname(os.path.abspath(path))
    if d:
        os.makedirs(d, exist_ok=True)

def build_or_load_token_stream(
    cache_path: str,
    dataset_name: str,
    dataset_config: str,
    split: str,
    tokenizer_name: str,
    max_rows: Optional[int] = None,
    add_eos: bool = True,
) -> List[int]:
    """Build or load tokenized stream from HF dataset."""

    if os.path.exists(cache_path):
        print(f"Loading cached tokens: {cache_path}")
        with open(cache_path, "rb") as f:
            hdr = f.read(_HDR_STRUCT.size)
        magic, eos_u32, n_tokens = _HDR_STRUCT.unpack(hdr)

        if magic != _MAGIC:
            raise ValueError(f"Bad cache magic: {magic}")

        mm = np.memmap(
            cache_path, mode="c", dtype=np.uint32,
            offset=_HDR_STRUCT.size, shape=(int(n_tokens),)
        )
        tokens = mm.tolist()
        print(f"Loaded {len(tokens):,} tokens")
        return tokens

    # Build from scratch
    print(f"Building token stream from {dataset_name}/{dataset_config}...")
    _ensure_dir(cache_path)

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    eos = tokenizer.eos_token_id if add_eos else None

    ds = load_dataset(dataset_name, dataset_config, split=split, streaming=True)
    if max_rows:
        ds = ds.take(max_rows)

    tokens = []
    for i, row in enumerate(tqdm(ds, desc="Tokenizing")):
        text = row.get("text", "")
        if len(text) < 10:
            continue
        toks = tokenizer.encode(text, add_special_tokens=False)
        tokens.extend(toks)
        if eos is not None:
            tokens.append(eos)

        if max_rows and i >= max_rows - 1:
            break

    # Write to cache
    with open(cache_path, "wb") as f:
        f.write(_HDR_STRUCT.pack(_MAGIC, eos or 0, len(tokens)))
        arr = np.array(tokens, dtype=np.uint32)
        f.write(arr.tobytes())

    print(f"Cached {len(tokens):,} tokens to {cache_path}")
    return tokens


# === Validation dataset ===
class StableValidationDataset(Dataset):
    def __init__(self, samples: List[Tuple[torch.Tensor, torch.Tensor]]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def build_validation_windows(
    token_stream: List[int],
    max_seq_len: int,
    stride_frac: float = 0.5,
    max_windows: int = 5_000,
) -> StableValidationDataset:
    """Chunk validation tokens into sliding windows."""

    T = max_seq_len
    stride = max(1, int(T * stride_frac))

    samples = []
    for start in range(0, len(token_stream) - T, stride):
        chunk = token_stream[start:start + T + 1]
        if len(chunk) < T + 1:
            break

        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        samples.append((x, y))

        if len(samples) >= max_windows:
            break

    print(f"Built {len(samples)} validation windows")
    return StableValidationDataset(samples)

print("✓ Data utilities loaded")

In [ ]:

#@title Load Training & Validation Data
#@markdown This will download and tokenize FineWeb. Takes ~10-20 min first time, then cached.

# Cache paths
train_cache = os.path.join(CACHE_DIR, "train_tokens.u32")
val_cache = os.path.join(CACHE_DIR, "val_tokens.u32")

# Load/build training tokens
train_tokens = build_or_load_token_stream(
    cache_path=train_cache,
    dataset_name=cfg.dataset_name,
    dataset_config=cfg.dataset_config,
    split="train",
    tokenizer_name=cfg.tokenizer_name,
    max_rows=10_000,  # Limit for Colab (use None for full dataset)
    add_eos=True,
)

# Load/build validation tokens
val_tokens = build_or_load_token_stream(
    cache_path=val_cache,
    dataset_name=cfg.dataset_name,
    dataset_config=cfg.dataset_config,
    split="train",  # FineWeb only has 'train' split
    tokenizer_name=cfg.tokenizer_name,
    max_rows=5_000,  # Smaller for validation
    add_eos=True,
)

# Build validation dataset
val_dataset = build_validation_windows(
    token_stream=val_tokens,
    max_seq_len=cfg.max_seq_len,
    stride_frac=0.5,
    max_windows=2_000,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(f"\n✓ Data loaded:")
print(f"  Train tokens: {len(train_tokens):,}")
print(f"  Val windows: {len(val_dataset):,}")

In [ ]:

#@title Training Utilities
#@markdown Learning rate scheduler, optimizer, and eval function.

# === LR Scheduler ===
class WarmupCosine:
    def __init__(self, optimizer, warmup_steps, total_steps, base_lr):
        self.opt = optimizer
        self.warmup = int(warmup_steps)
        self.total = int(total_steps)
        self.base = float(base_lr)
        self.step_num = 0

    def step(self):
        self.step_num += 1
        if self.step_num <= self.warmup:
            lr = self.base * self.step_num / max(1, self.warmup)
        else:
            prog = (self.step_num - self.warmup) / max(1, self.total - self.warmup)
            lr = self.base * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))

        for g in self.opt.param_groups:
            g["lr"] = lr
        return lr


# === Evaluation Function ===
@torch.no_grad()
def evaluate(model, val_loader, max_batches=None):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for i, (x, y) in enumerate(val_loader):
        if max_batches and i >= max_batches:
            break

        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(avg_loss, 20))  # Cap to prevent overflow

    return avg_loss, ppl


# === Iterable Token Dataset ===
class TokenIterableDataset(torch.utils.data.IterableDataset):
    def __init__(self, tokens, seq_len, infinite=True):
        self.tokens = tokens
        self.seq_len = seq_len
        self.infinite = infinite

    def __iter__(self):
        while True:
            # Random start position
            max_start = len(self.tokens) - self.seq_len - 1
            if max_start <= 0:
                raise ValueError("Not enough tokens")

            start = torch.randint(0, max_start, (1,)).item()
            chunk = self.tokens[start:start + self.seq_len + 1]

            x = torch.tensor(chunk[:-1], dtype=torch.long)
            y = torch.tensor(chunk[1:], dtype=torch.long)

            yield x, y

            if not self.infinite:
                break

print("✓ Training utilities loaded")

In [ ]:

#@title Initialize Model & Optimizer

# Build model
model = build_model_from_cfg(cfg)
model = model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    betas=(0.9, 0.95),
)

# Scheduler
scheduler = WarmupCosine(
    optimizer,
    warmup_steps=cfg.warmup_steps,
    total_steps=cfg.total_steps,
    base_lr=cfg.learning_rate,
)

# Training dataloader
train_dataset = TokenIterableDataset(
    tokens=train_tokens,
    seq_len=cfg.max_seq_len,
    infinite=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=MICRO_BATCH_SIZE,
    num_workers=0,
)

# Setup AMP
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda") if use_amp else None

print(f"✓ Model initialized:")
print(f"  Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(f"  AMP: {use_amp}")
print(f"  Optimizer: AdamW (lr={cfg.learning_rate}, wd={cfg.weight_decay})")

In [ ]:

#@title Training Loop
#@markdown Run training. Monitor loss and save checkpoints.

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, TAG), exist_ok=True)

# Training state
global_step = 0
best_val_loss = float('inf')
train_iter = iter(train_loader)

model.train()
optimizer.zero_grad()

pbar = tqdm(total=cfg.total_steps, desc="Training")

while global_step < cfg.total_steps:

    # === Gradient Accumulation Loop ===
    batch_loss = 0.0
    for micro_step in range(GRAD_ACCUM_STEPS):
        x, y = next(train_iter)
        x, y = x.to(device), y.to(device)

        # Forward
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            loss = loss / GRAD_ACCUM_STEPS  # Scale for accumulation

        # Backward
        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        batch_loss += loss.item()

    # === Optimizer Step ===
    if use_amp:
        scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

    if use_amp:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()

    optimizer.zero_grad()
    lr = scheduler.step()

    global_step += 1
    pbar.update(1)

    # === Logging ===
    if global_step % LOG_INTERVAL == 0:
        ppl = math.exp(min(batch_loss, 20))
        pbar.set_postfix({
            'loss': f'{batch_loss:.3f}',
            'ppl': f'{ppl:.1f}',
            'lr': f'{lr:.2e}'
        })

    # === Evaluation ===
    if global_step % EVAL_INTERVAL == 0:
        val_loss, val_ppl = evaluate(model, val_loader, max_batches=50)

        print(f"\nStep {global_step}: val_loss={val_loss:.3f}, val_ppl={val_ppl:.1f}")

        # Save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            ckpt_path = os.path.join(OUTPUT_DIR, TAG, "best.pt")
            torch.save({
                'model': model.state_dict(),
                'cfg': asdict(cfg),
                'step': global_step,
                'val_loss': val_loss,
            }, ckpt_path)
            print(f"✓ Saved best checkpoint: {ckpt_path}")

        model.train()

    # === Save Last ===
    if global_step % SAVE_LAST_EVERY == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, TAG, "last.pt")
        torch.save({
            'model': model.state_dict(),
            'cfg': asdict(cfg),
            'step': global_step,
            'optimizer': optimizer.state_dict(),
        }, ckpt_path)

pbar.close()
print(f"\n✓ Training complete! Best val loss: {best_val_loss:.3f}")